In [3]:
import sys, os

_proj_db = r'C:\Users\marrocol\AppData\Local\miniforge3\envs\mswe-gnn\Lib\site-packages\pyproj\proj_dir\share\proj'
os.environ.setdefault('PROJ_DATA', _proj_db)
os.environ.setdefault('PROJ_LIB',  _proj_db)

try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset
from utils.visualization import PlotRollout
from training.train import LightningTrainer

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')
print('Repo root:', REPO_ROOT)

Repo root: c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg


In [ ]:
api = wandb.Api()

ENTITY    = "mmarrocolo-tu-delft"
PROJECT   = "mswe-gnn-ahr-K-calib-3scales"
SWEEP_IDS = ["a97bvnpt", "j1kb8vdo"]  # a97bvnpt: original grid (3/6 succeeded); j1kb8vdo: retry
                                       

runs = []
for sweep_id in SWEEP_IDS:
    sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")
    sweep_runs = [r for r in sweep.runs if r.summary.get('val_loss') is not None]
    print(f"{sweep_id}: {len(sweep_runs)} finished run(s)")
    runs.extend(sweep_runs)

# for sweep_id in SWEEP_IDS:
#     sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")
#     sweep_runs = [r for r in sweep.runs if r.summary.get('val_CSI_03') is not None]
#     print(f"{sweep_id}: {len(sweep_runs)} finished run(s)")
#     runs.extend(sweep_runs)

best_run = min(runs, key=lambda r: r.summary['val_loss'])
#best_run = max(runs, key=lambda r: r.summary['val_CSI_03'])

print(f"\nBest run : {best_run.name}  (id: {best_run.id}, sweep: {best_run.sweep.id if best_run.sweep else '?'})")
print(f"val_loss     : {best_run.summary.get('val_loss',    float('nan')):.4f}")
print(f"val_CSI_005  : {best_run.summary.get('val_CSI_005', float('nan')):.4f}")
print(f"val_CSI_03   : {best_run.summary.get('val_CSI_03',  float('nan')):.4f}")
print(f"\nHyperparameters:")
for k, v in sorted(best_run.config.items()):
    print(f"  {k}: {v}")


a97bvnpt: 3 finished run(s)
j1kb8vdo: 2 finished run(s)

Best run : rollout5_hid32_[1,1,9,4,3]  (id: 2bk1mda5, sweep: j1kb8vdo)
val_loss     : 0.2732
val_CSI_005  : 0.7914
val_CSI_03   : 0.8490

Hyperparameters:
  K: [1, 1, 9, 4, 3]
  dataset_parameters: {'seed': 0, 'val_prcnt': 0, 'train_size': 4, 'temporal_res': 60, 'dataset_folder': 'database/datasets', 'test_dataset_name': 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_3scales', 'train_dataset_name': 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_3scales_multisim'}
  lr_info: {'T_max': 1000, 'gamma': 0.7, 'eta_min': 1e-06, 'scheduler': 'cosine', 'step_size': 10, 'weight_decay': 0, 'learning_rate': 0.0005177909168244793}
  models: {'K': [1, 1, 5, 3, 2], 'seed': 666, 'with_WL': True, 'edge_mlp': True, 'normalize': True, 'mlp_layers': 3, 'model_type': 'MSGNN', 'hid_features': 32, 'with_gradient': True, 'gnn_activation': 'tanh', 'mlp_activation': 'prelu', 'learned_pooling': False, 'skip_connections': True, 'learne